# Lab 34: Head-to-head RAG pattern evaluation

Run static, CRAG, Self-RAG, and Graph RAG over one shared corpus and eval set, and score them with the Batch 69 evaluation framework. Fill in the `TODO` cells; reference implementation in `solution/`.

The result motivates Lab 35 (the adaptive router): each pattern wins a different query category.

## Step 0: Setup

In [ ]:
import json
import os
import pathlib
import re
from collections import defaultdict
from dotenv import load_dotenv

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break
assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY")
PROVIDER = "openai"
MODEL = {"openai": "gpt-4o-mini", "anthropic": "claude-haiku-4-5-20251001"}[PROVIDER]
print(f"Using {PROVIDER} / {MODEL}")

In [ ]:
def chat(messages, temperature=0.0):
    if PROVIDER == "openai":
        from openai import OpenAI
        r = OpenAI().chat.completions.create(model=MODEL, messages=messages, temperature=temperature)
        return r.choices[0].message.content or ""
    from anthropic import Anthropic
    system = next((m["content"] for m in messages if m["role"]=="system"), "")
    ns = [m for m in messages if m["role"]!="system"]
    r = Anthropic().messages.create(model=MODEL, system=system, messages=ns, max_tokens=1024, temperature=temperature)
    return "".join(b.text for b in r.content if hasattr(b,"text"))

def chat_token(messages, allowed):
    raw = chat(messages).strip().lower()
    for tok in allowed:
        if re.search(rf"\\b{re.escape(tok.lower())}\\b", raw):
            return tok
    return allowed[-1]

## Step 1: Shared index over the entity-rich corpus

All flat patterns retrieve over the same chunks; Graph RAG (Step 3) uses the same source docs.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Shared corpus: Lab 33's entity-rich ecosystem (so Graph RAG can shine and the
# flat patterns retrieve over the same source documents).
CORPUS_DIR = pathlib.Path("../33-graph-rag-from-scratch/corpus")
def approx_tokens(t): return int(len(t.split())/0.75)
def split_paras(t): return [p.strip() for p in re.split(r"\n\s*\n", t) if p.strip()]
def split_sents(t): return [p.strip() for p in re.split(r"(?<=[.!?])\s+", t) if p.strip()]
def chunk_text(text, target=120):
    out,cur,ct=[],[],0
    for para in split_paras(text):
        pt=approx_tokens(para)
        if ct + pt > target and cur:
            out.append("\n\n".join(cur))
            cur, ct = [], 0
        if pt>target:
            for s in split_sents(para):
                st=approx_tokens(s)
                if ct + st > target and cur:
                    out.append(" ".join(cur))
                    cur, ct = [], 0
                cur.append(s)
                ct += st
        else:
            cur.append(para)
            ct += pt
    if cur:
        out.append("\n\n".join(cur))
    return out

docs = []
all_chunks = []
for path in sorted(CORPUS_DIR.glob("*.md")):
    if path.name == "README.md":
        continue
    body = path.read_text()
    title = body.splitlines()[0].lstrip("# ").strip()
    docs.append({"doc_id": path.stem, "title": title, "text": body})
    for i, ch in enumerate(chunk_text(body)):
        all_chunks.append({"doc_id": path.stem, "chunk_id": f"{path.stem}#{i}", "text": ch})

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
E = embedder.encode([c["text"] for c in all_chunks], normalize_embeddings=True,
                    convert_to_numpy=True, show_progress_bar=False)
def search(query, k=5):
    q = embedder.encode([query], normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)[0]
    s = E @ q
    return [{**all_chunks[i], "score": float(s[i])} for i in np.argsort(s)[::-1][:k]]
print(f"{len(docs)} docs, {len(all_chunks)} chunks indexed")

## Step 2: Three flat pipelines (static, CRAG, Self-RAG)

Each condensed to a common interface: `pipe(query) -> {answer, retrieved_docs, retrieved}`.

In [ ]:
def _gen(query, context, allow_abstain=True):
    sys = ("Answer using only the evidence. Cite doc ids in [brackets]. "
           + ("If the evidence does not contain the answer, reply exactly "
              "'INSUFFICIENT EVIDENCE'." if allow_abstain else ""))
    return chat([{"role":"system","content":sys},
                 {"role":"user","content":f"Evidence:\n{context}\n\nQuestion: {query}"}])

def pipe_static(query):
    """TODO: retrieve top-k, build context, generate. Return
    {answer, retrieved_docs, retrieved=True}."""
    raise NotImplementedError

def pipe_crag(query):
    """TODO: retrieve, grade verdict (correct/ambiguous/incorrect); abstain on
    incorrect, else generate. Return dict as above plus verdict."""
    raise NotImplementedError

def pipe_self_rag(query):
    """TODO: decide_retrieve; if no_retrieve answer from general knowledge with
    retrieved=False; else retrieve + generate."""
    raise NotImplementedError

## Step 3: The Graph RAG pipeline

Builds the knowledge graph once, then routes global (map-reduce) vs local (traversal). This is the expensive setup; it runs once, not per query.

In [ ]:
# Graph RAG pipeline. Builds the graph ONCE (expensive), then routes global/local.
import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities

def _extract(doc):
    out = chat([
        {"role":"system","content":"Extract entities and relationships as JSON only: "
         "{\"entities\":[{\"name\":str}],\"relations\":[{\"source\":str,\"target\":str,\"desc\":str}]}. "
         "Use concise canonical names."},
        {"role":"user","content":f"{doc['title']}\n\n{doc['text'][:3000]}"}])
    out = re.sub(r"^```(json)?|```$","",out.strip(),flags=re.MULTILINE).strip()
    try:
        return json.loads(out)
    except Exception:
        m = re.search(r"\{.*\}", out, re.DOTALL)
        return json.loads(m.group(0)) if m else {"entities": [], "relations": []}

def build_graph():
    G = nx.Graph()
    for d in docs:
        er = _extract(d)
        for e in er.get("entities",[]):
            nm = e.get("name","").strip().lower()
            if not nm:
                continue
            if G.has_node(nm):
                G.nodes[nm]["docs"].add(d["doc_id"])
            else:
                G.add_node(nm, docs={d["doc_id"]})
        for r in er.get("relations",[]):
            s,t = r.get("source","").strip().lower(), r.get("target","").strip().lower()
            if s and t and s != t:
                G.add_edge(s, t, desc=r.get("desc", ""))
    return G

print("Building knowledge graph (one LLM call per doc)...")
G = build_graph()
comms = list(greedy_modularity_communities(G)) if G.number_of_edges() else [set(G.nodes())]
comm_summaries = []
for i,c in enumerate(comms):
    edges = [(u,v,G.edges[u,v].get("desc","")) for u,v in G.subgraph(c).edges()]
    el = "\n".join(f"- {u} -> {v}: {d}" for u,v,d in edges) or "(none)"
    summ = chat([{"role":"system","content":"Summarize this cluster of related concepts in 2-3 sentences."},
                 {"role":"user","content":f"Entities: {', '.join(sorted(c))}\n\nRelationships:\n{el}"}])
    comm_summaries.append({"id":i,"entities":sorted(c),"summary":summ})
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges, {len(comms)} communities")

def pipe_graph(query):
    kind = chat_token([{"role":"system","content":"Is this a GLOBAL question (themes/patterns across "
        "the whole corpus) or LOCAL (a specific entity)? Answer one word: global or local."},
        {"role":"user","content":query}], allowed=["global","local"])
    if kind == "global":
        partials=[]
        for cs in comm_summaries:
            p = chat([{"role":"system","content":"Using only this summary, give a partial answer or 'NONE'."},
                      {"role":"user","content":f"Summary: {cs['summary']}\n\nQuery: {query}"}])
            if "none" not in p.strip().lower()[:8]:
                partials.append(p)
        ans = chat([{"role":"system","content":"Synthesize the partials into one answer."},
                    {"role":"user","content":"Partials:\n"+"\n\n".join(partials)+f"\n\nQuery: {query}"}])
        # provenance: which docs did the contributing entities come from
        rdocs = sorted({d for cs in comm_summaries for e in cs["entities"]
                        if e in G for d in G.nodes[e]["docs"]})
        return {"answer": ans, "retrieved_docs": rdocs[:6], "retrieved": True, "route":"global"}
    # local
    q = query.lower()
    seeds = [n for n in G.nodes() if n in q]
    if not seeds and G.number_of_nodes():
        seeds = [max(G.nodes(), key=lambda n: G.degree(n))]
    nodes=set(seeds)
    for n in list(seeds):
        nodes |= set(G.neighbors(n))
    edges=[(u,v,G.edges[u,v].get("desc","")) for u,v in G.subgraph(nodes).edges()]
    ctx="\n".join(f"- {u} -> {v}: {d}" for u,v,d in edges) or "(none)"
    ans = chat([{"role":"system","content":"Answer using only these relationships."},
                {"role":"user","content":f"Subgraph:\n{ctx}\n\nQuery: {query}"}])
    rdocs = sorted({d for n in nodes if n in G for d in G.nodes[n]["docs"]})
    return {"answer": ans, "retrieved_docs": rdocs[:6], "retrieved": True, "route":"local"}

PIPELINES = {"static": pipe_static, "crag": pipe_crag, "self_rag": pipe_self_rag, "graph": pipe_graph}

## Step 4: Scoring (the Batch 69 framework, applied)

Retrieval recall on the generation side; answer correctness via expected tokens; abstention correctness for off-corpus queries.

In [ ]:
# --- Scoring (the Batch 69 framework, applied) ---
# Retrieval side: doc-level recall (of the labeled relevant docs, how many were
#   retrieved). Generation side: answer correctness (expected tokens present) and
#   abstention correctness (did off-corpus queries get refused?).

ABSTAIN_MARKERS = ["insufficient evidence", "does not contain", "not in the", "cannot answer",
                   "no information", "not enough information", "don't have", "do not have"]

def abstained(answer: str) -> bool:
    a = answer.lower()
    return any(m in a for m in ABSTAIN_MARKERS)

def doc_recall(retrieved_docs, relevant_docs) -> float:
    if not relevant_docs:
        return float("nan")  # N/A for off-corpus / parametric
    hit = len(set(retrieved_docs) & set(relevant_docs))
    return hit / len(relevant_docs)

def answer_correct(answer: str, item: dict) -> bool:
    beh = item["expected_behavior"]
    if beh == "abstain":
        return abstained(answer)              # correct = refused
    # answer / answer_without_retrieval: all expected tokens present, and not abstained
    if abstained(answer):
        return False
    a = answer.lower()
    return all(tok.lower() in a for tok in item["expected_contains"])

with open("./eval_set.jsonl") as f:
    eval_set = [json.loads(line) for line in f]
print(f"Loaded {len(eval_set)} eval queries across "
      f"{len({e['category'] for e in eval_set})} categories")

## Step 5: Run the harness

Every pattern over every query.

In [ ]:
def run_harness(pipelines, eval_set):
    """TODO: for each pattern and each query, call the pipeline and record
    {pattern, id, category, behavior, recall, correct, retrieved, answer}.
    Use doc_recall(...) and answer_correct(...)."""
    raise NotImplementedError

records = run_harness(PIPELINES, eval_set)
print(f"Ran {len(records)} (pattern x query) evaluations")

## Step 6: The comparison table

Answer correctness by pattern x category, plus retrieval recall and off-corpus abstention.

In [ ]:
# --- Comparison table: answer correctness by pattern x category ---
import math
patterns = list(PIPELINES.keys())
cats = ["specific-lookup","paraphrase","multi-hop","global-theme","off-corpus","parametric"]

def acc(pattern, cat):
    rs = [r for r in records if r["pattern"]==pattern and r["category"]==cat]
    return sum(r["correct"] for r in rs)/len(rs) if rs else float("nan")

hdr = "pattern".ljust(10) + "".join(c[:13].rjust(15) for c in cats) + "   OVERALL"
print(hdr)
print("-" * len(hdr))
for p in patterns:
    row = p.ljust(10)
    for c in cats:
        v = acc(p, c)
        row += ("  n/a" if math.isnan(v) else f"{v:.2f}").rjust(15)
    allr = [r for r in records if r["pattern"]==p]
    row += f"{sum(r['correct'] for r in allr)/len(allr):.2f}".rjust(11)
    print(row)

print("\n--- Retrieval doc-recall (flat patterns; graph routes differently) ---")
for p in patterns:
    rs = [r["recall"] for r in records if r["pattern"]==p and not math.isnan(r["recall"])]
    print(f"  {p:10} mean doc-recall = {sum(rs)/len(rs):.2f}" if rs else f"  {p:10} n/a")

print("\n--- Off-corpus abstention (did each pattern refuse the 3 unanswerable queries?) ---")
for p in patterns:
    oc = [r for r in records if r["pattern"]==p and r["category"]=="off-corpus"]
    print(f"  {p:10} abstained correctly on {sum(r['correct'] for r in oc)}/{len(oc)}")

## Step 7: Read the result

The instructive finding and what it motivates.

In [ ]:
# Expected shape of the result (your exact numbers vary with the LLM):
#
#  - STATIC wins specific-lookup and paraphrase, but ANSWERS off-corpus queries
#    (no abstention mechanism) -> it fabricates, scoring 0 on off-corpus.
#  - CRAG matches static on answerable queries AND abstains on off-corpus -> it
#    wins the off-corpus column.
#  - SELF_RAG skips retrieval on parametric queries -> it wins the parametric
#    column (static/crag try to ground a general-knowledge question in the corpus).
#  - GRAPH wins global-theme and is competitive on multi-hop, because those
#    answers live in the relationships BETWEEN documents, not in any single chunk.
#
# The headline: NO SINGLE PATTERN DOMINATES. Each wins the category it was
# designed for. That is precisely the empirical case for an ADAPTIVE ROUTER
# (Lab 35): classify the query, then dispatch to the pattern that wins its kind.
print("See Lab 35 (Adaptive RAG router) for the pattern this comparison motivates.")

## What you built

A head-to-head harness that scores four RAG patterns on one corpus and eval set, sliced by query category. The result is the empirical case for adaptive routing: each pattern wins the category it was built for, and none dominates. [Lab 35](../35-adaptive-rag-router/) turns that finding into a router.

**Where this simplifies:** the pattern pipelines are condensed (full versions in Labs 06/31/32/33); answer-correctness uses token presence rather than full LLM-judge faithfulness (an optional extension — wire in the `eval_gate` and an LLM judge from the evaluation framework); the corpus is small enough to read by hand, which is a feature for a teaching harness and a limit for strong claims.